
# Wossist — RFM Customer Segmentation

## Project Overview

This notebook develops an RFM-based customer segmentation for **Wossist**, a fictional insurance company operating across Travel, Pet and Home insurance products.

The analysis focuses on customers with at least one observed policy purchase, examining their purchasing behaviour through Recency, Frequency and Monetary Value.

The objective is to develop an interpretable segmentation methodology adapted to Wossist's observed transactional patterns, providing a foundation for differentiated CRM and marketing analysis.

RFM segmentation represents the first analytical component of the broader **Customer Segmentation & Profiling** project. Additional business-oriented profiling dimensions are developed separately.

All data used in this analysis is entirely synthetic.

## Business Question

**How can Wossist classify Purchasing Customers based on their observed purchasing behaviour and economic value to support differentiated CRM actions?**

The analysis addresses this question by:

- Calculating customer-level Recency, Frequency and Monetary Value.
- Exploring purchasing distributions and defining appropriate scoring rules.
- Combining RFM scores into interpretable customer segments.
- Validating the resulting segments and their business characteristics.

The analysis prioritizes explainable, data-driven segmentation rules rather than assuming that conventional RFM classifications are automatically suitable for Wossist's insurance business.


## RFM Methodological Approach

RFM is used as the primary transactional segmentation dimension of the project.

The analysis is performed at the overall customer level rather than separately for each business line. Therefore, Recency, Frequency and Monetary Value describe the customer's complete purchasing relationship with Wossist.

This choice intentionally separates transactional behaviour from product preference:

- **RFM** describes the strength and value of the purchasing relationship;
- **Product Profile** describes what the customer purchases;
- the combination of both dimensions provides a richer interpretation than either dimension alone.

This distinction is particularly relevant for Wossist because its business lines have different purchasing patterns and premium levels. For example, Travel policies may generate relatively frequent but lower-value transactions, while Home policies may generate fewer but higher-value transactions.


These differences are retained within the overall RFM metrics and should be considered when interpreting the resulting scores and segments. Product-specific purchasing patterns will be explored separately through Customer Business Profiling.


This allows the segmentation methodology to be driven by both established RFM principles and the observed characteristics of the Wossist business.

### Economic Value Assumption

Monetary Value is based on the total insurance premium generated by each customer.

Premium is therefore used as a proxy for customer economic value. The analysis does not model product-level costs, claims profitability or differences in contribution margins across business lines.

For the purpose of this project, differences in product-level margins are considered outside the analytical scope. As a result, customers generating higher total premium are treated as generating higher observed economic value for Wossist.

This assumption keeps the segmentation focused on observable customer and purchasing behaviour while avoiding the introduction of synthetic profitability variables that are not available in the source data.

## Data Loading and Preparation


The following section imports the Python libraries and source datasets required for the RFM analysis.

The data is retrieved directly from the Wossist project repository to ensure a reproducible analytical workflow.


In [4]:
import pandas as pd
import numpy as np
import plotly.express as px

BASE_URL = (
    "https://raw.githubusercontent.com/"
    "AntonioPennacchia/customer-analytics/main/"
    "insurance-customer-analytics/dataset/"
)

customers = pd.read_csv(BASE_URL + "customers.csv.gz")
policies = pd.read_csv(BASE_URL + "policies.csv.gz")

In [5]:
# Store all loaded datasets in a dictionary for consistent validation
# and exploratory checks throughout the notebook
datasets = {
    "customers": customers,
    "policies": policies
}

# Summarize the size of each source dataset
dataset_summary = pd.DataFrame({
    "Rows": [len(df) for df in datasets.values()],
    "Columns": [len(df.columns) for df in datasets.values()]
}, index=datasets.keys())

dataset_summary

,Rows,Columns
customers,631079,10
policies,223292,16


## Data Sources & Analytical Population


The Wossist dataset includes both registered users who have purchased insurance policies and users without an observed purchase.

RFM analysis is applicable only to **Purchasing Customers**, defined as registered users with at least one observed policy purchase.

The complete registered user population is examined to establish the analytical scope, while RFM metrics and scores are calculated exclusively for Purchasing Customers.

Non-Purchasing Users are addressed separately in the Customer Business Profiling notebook.


In [6]:
# Define date columns for each dataset
date_columns = {
    "customers": ["birth_date", "registration_date"],
    "policies": ["purchase_date", "policy_start_date", "policy_end_date"]
}

# Convert date fields to pandas datetime format
for dataset_name, columns in date_columns.items():
    for column in columns:
        datasets[dataset_name][column] = pd.to_datetime(
            datasets[dataset_name][column],
            errors="coerce"
        )

print("Date columns converted successfully.")

Date columns converted successfully.


### Analytical Time Horizon


The synthetic Wossist dataset covers the period from 2024 to 2026.

The analysis uses a fixed observation date of **31 December 2026** to calculate Recency consistently across all Purchasing Customers.

Using a fixed observation date also ensures reproducibility: RFM results do not change depending on when the notebook is executed.


In [7]:
# Fixed observation date used for time-dependent segmentation features
ANALYSIS_DATE = pd.Timestamp("2026-12-31")

# Inspect the main customer and purchasing time horizons
print(f"Analysis date: {ANALYSIS_DATE.date()}")
print(
    f"Registration period: "
    f"{customers['registration_date'].min().date()} → "
    f"{customers['registration_date'].max().date()}"
)
print(
    f"Purchase period: "
    f"{policies['purchase_date'].min().date()} → "
    f"{policies['purchase_date'].max().date()}"
)

Analysis date: 2026-12-31
Registration period: 2024-01-01 → 2026-12-31
Purchase period: 2024-01-14 → 2026-12-31


In [8]:
# Identify registered users with at least one policy purchase
purchasing_customer_ids = policies["customer_id"].dropna().unique()

# Flag purchasing customers within the complete registered user base
customers["is_purchasing_customer"] = customers["customer_id"].isin(
    purchasing_customer_ids
)

# Summarize the analytical populations
population_summary = pd.DataFrame({
    "Population": [
        "Registered Users",
        "Purchasing Customers",
        "Non-Purchasing Users"
    ],
    "Users": [
        customers["customer_id"].nunique(),
        customers.loc[
            customers["is_purchasing_customer"], "customer_id"
        ].nunique(),
        customers.loc[
            ~customers["is_purchasing_customer"], "customer_id"
        ].nunique()
    ]
})

population_summary["Share"] = (
    population_summary["Users"]
    / customers["customer_id"].nunique()
)

population_summary.style.format({
    "Users": "{:,.0f}",
    "Share": "{:.1%}"
})

,Population,Users,Share
0,Registered Users,"631,079",100.0%
1,Purchasing Customers,"198,073",31.4%
2,Non-Purchasing Users,"433,006",68.6%


### Population Overview


The Wossist customer base contains **631,079 registered users**, of whom **198,073 (31.4%)** have purchased at least one policy during the observation period.

The remaining **433,006 users (68.6%)** have no observed policy purchases and are therefore excluded from RFM scoring.

These users will be included in the subsequent Customer Business Profiling analysis, which extends the analytical framework beyond purchasing behaviour.


## Data Quality Check

Before starting the RFM analysis, the key transactional fields required to calculate Recency, Frequency and Monetary Value are checked for completeness.

The validation focuses on the fields that directly affect the segmentation methodology rather than performing an exhaustive audit of all source attributes.

In [9]:
# Check completeness of fields required for RFM analysis
rfm_fields = [
    "customer_id",
    "purchase_date",
    "premium_amount"
]

rfm_quality = policies[rfm_fields].isna().sum().to_frame("Missing Values")

rfm_quality

,Missing Values
customer_id,0
purchase_date,0
premium_amount,0


### Data Quality Conclusion

The fields required for the RFM analysis are complete, providing a consistent basis for customer-level feature engineering.

Product-specific missing values are treated as part of the natural structure of the data and are not considered data quality issues unless they affect a specific analytical use case.

The data preparation and validation phase is therefore complete.

## RFM Analysis

RFM analysis is used to describe the customer's purchasing relationship with Wossist through three transactional dimensions:

- **Recency** — how recently the customer made a purchase;
- **Frequency** — how many policies the customer has purchased;
- **Monetary Value** — the total premium generated by the customer.

The three dimensions provide complementary information about the customer's relationship with Wossist. Recency captures the temporal proximity of the latest purchase, Frequency reflects the observed purchasing intensity, while Monetary Value captures the economic value generated over the observation period.

For this project, RFM is calculated at customer level using all observed policy purchases up to the fixed analysis date of **31 December 2026**.

The resulting RFM features will initially be analysed as continuous variables. Scoring thresholds and customer segments will be defined only after examining their distributions and the specific purchasing patterns of the Wossist customer base.

In [10]:
# Aggregate policy transactions at customer level
rfm_base = (
    policies
    .groupby("customer_id")
    .agg(
        last_purchase_date=("purchase_date", "max"),
        frequency=("policy_id", "nunique"),
        monetary_value=("premium_amount", "sum")
    )
    .reset_index()
)

# Calculate the number of days since the customer's most recent purchase
rfm_base["recency_days"] = (
    ANALYSIS_DATE - rfm_base["last_purchase_date"]
).dt.days

# Reorder columns to follow the conventional RFM structure
rfm_base = rfm_base[
    [
        "customer_id",
        "recency_days",
        "frequency",
        "monetary_value",
        "last_purchase_date"
    ]
]

rfm_base.head()

,customer_id,recency_days,frequency,monetary_value,last_purchase_date
0,C000001,380,4,635.64,2025-12-16
1,C000002,887,1,162.95,2024-07-27
2,C000003,351,3,373.98,2026-01-14
3,C000004,973,1,18.46,2024-05-02
4,C000005,917,1,14.29,2024-06-27


### RFM Customer Base

RFM features are calculated for all customers with at least one observed policy purchase.

The resulting analytical table contains one row per Purchasing Customer and provides the three continuous RFM measures that will be used for the subsequent distribution analysis and scoring methodology.

In [11]:
# Validate the size and completeness of the RFM customer-level table
rfm_summary = pd.DataFrame({
    "Metric": [
        "RFM Customers",
        "Missing Recency",
        "Missing Frequency",
        "Missing Monetary Value"
    ],
    "Value": [
        rfm_base["customer_id"].nunique(),
        rfm_base["recency_days"].isna().sum(),
        rfm_base["frequency"].isna().sum(),
        rfm_base["monetary_value"].isna().sum()
    ]
})

rfm_summary

,Metric,Value
0,RFM Customers,198073
1,Missing Recency,0
2,Missing Frequency,0
3,Missing Monetary Value,0


### RFM Distribution Analysis

Before defining the RFM scoring methodology, the distributions of Recency, Frequency and Monetary Value are examined separately.

This step is particularly important because the three measures may have different distributions and levels of concentration across the Wossist customer base.

The analysis will be used to identify:

- the typical range of values for each dimension;
- the presence of highly concentrated or skewed distributions;
- potential outliers;
- whether standard percentile-based scoring can be applied consistently across the three dimensions.

No customer segment is assigned at this stage. The purpose of this analysis is to inform the scoring methodology that follows.

In [12]:
# Calculate descriptive statistics for the three RFM dimensions
rfm_distribution = (
    rfm_base[
        ["recency_days", "frequency", "monetary_value"]
    ]
    .describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
    .T
)

rfm_distribution

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
recency_days,198073.0,286.546344,241.477409,0.0,1.00,7.0,105.0,203.00,488.0,809.000,934.0000,1082.00
frequency,198073.0,1.127322,0.504644,1.0,1.00,1.0,1.0,1.00,1.0,2.000,4.0000,7.00
monetary_value,198073.0,138.164865,197.900799,10.0,11.78,13.7,19.5,35.87,170.0,587.734,883.8068,1561.07


In [13]:
# Examine the most common purchase frequencies
frequency_distribution = (
    rfm_base["frequency"]
    .value_counts()
    .sort_index()
    .head(15)
    .to_frame("Customers")
)

frequency_distribution

,Customers
frequency,
1,182966
2,8319
3,4165
4,1996
5,559
6,62
7,6


In [14]:
# Recency distribution
fig = px.histogram(
    rfm_base,
    x="recency_days",
    nbins=35,
    title="Recency Distribution",
    labels={
        "recency_days": "Days since last purchase",
        "count": "Customers"
    },
    template="plotly_white"
)

fig.add_vline(
    x=rfm_base["recency_days"].median(),
    line_dash="dash",
    annotation_text=f"Median: {rfm_base['recency_days'].median():.0f} days",
    annotation_position="top right"
)

fig.update_layout(
    title_font_size=20,
    title_font_color="#222222",
    xaxis_title_font_size=13,
    yaxis_title_font_size=13,
    showlegend=False,
    height=450,
    margin=dict(l=50, r=30, t=70, b=50)
)

fig.show()

In [15]:
# Frequency distribution
frequency_plot = (
    rfm_base["frequency"]
    .value_counts()
    .sort_index()
    .reset_index()
)

frequency_plot.columns = ["frequency", "customers"]

fig = px.bar(
    frequency_plot,
    x="frequency",
    y="customers",
    text="customers",
    title="Frequency Distribution",
    labels={
        "frequency": "Number of policies purchased",
        "customers": "Customers"
    },
    template="plotly_white"
)

fig.update_traces(
    texttemplate="%{text:,}",
    textposition="outside",
    hovertemplate=(
        "Policies purchased: %{x}<br>"
        "Customers: %{y:,}<extra></extra>"
    )
)

fig.update_layout(
    title_font_size=20,
    title_font_color="#222222",
    xaxis=dict(
        dtick=1
    ),
    showlegend=False,
    height=450,
    margin=dict(l=50, r=30, t=70, b=50)
)

fig.show()

In [16]:
# Monetary value distribution
fig = px.histogram(
    rfm_base,
    x="monetary_value",
    nbins=45,
    title="Monetary Value Distribution",
    labels={
        "monetary_value": "Total premium (€)",
        "count": "Customers"
    },
    template="plotly_white"
)

fig.add_vline(
    x=rfm_base["monetary_value"].median(),
    line_dash="dash",
    annotation_text=f"Median: €{rfm_base['monetary_value'].median():,.2f}",
    annotation_position="top right"
)

fig.update_layout(
    title_font_size=20,
    title_font_color="#222222",
    xaxis_title_font_size=13,
    yaxis_title_font_size=13,
    showlegend=False,
    height=450,
    margin=dict(l=50, r=30, t=70, b=50)
)

fig.show()


### RFM Distribution Findings

The exploratory analysis highlights different distribution patterns across the three RFM dimensions.

**Recency**

Recency shows a relatively broad distribution, with a median of 203 days and an interquartile range of 105–488 days. This provides sufficient variability to distinguish customers based on how recently they purchased a policy.

**Frequency**

Frequency is highly concentrated on single-policy customers. The median and 75th percentile are both equal to 1, while the 95th percentile is 2 and the maximum observed frequency is 7.

This concentration makes conventional percentile-based scoring unsuitable for Frequency, as multiple percentile thresholds would correspond to the same number of purchases.

**Monetary Value**

Monetary Value is strongly right-skewed. The median customer has generated €35.87 in premium, compared with an average of approximately €138.16. The 75th percentile reaches €170, while the 95th percentile exceeds €587.

This reflects substantial differences in observed premium generation across customers, potentially influenced by product mix and purchasing frequency.

**Methodological Implications**

The three dimensions should not necessarily use identical scoring rules:

- Recency can be evaluated using percentile-based thresholds.
- Frequency requires discrete thresholds reflecting the observed concentration of policy counts.
- Monetary Value can be evaluated using percentile-based thresholds, while acknowledging its asymmetric distribution.

The scoring methodology will therefore combine statistical criteria with business-specific considerations rather than applying a uniform scoring approach to all three dimensions.



### RFM Scoring Methodology

The RFM scoring methodology is adapted to the observed characteristics of the Wossist customer base.

Rather than applying identical scoring rules to all three dimensions, the approach combines discrete thresholds for Frequency with percentile-based thresholds for Recency and Monetary Value.

Each dimension is scored from 1 to 5, with higher scores representing more recent purchases, greater purchasing frequency or higher accumulated premium.


In [17]:

# Examine frequency distribution and customer concentration
frequency_scoring_analysis = (
    rfm_base["frequency"]
    .value_counts()
    .sort_index()
    .rename_axis("Policies Purchased")
    .reset_index(name="Customers")
)

frequency_scoring_analysis["Customer Share"] = (
    frequency_scoring_analysis["Customers"]
    / len(rfm_base)
)

frequency_scoring_analysis["Cumulative Share"] = (
    frequency_scoring_analysis["Customer Share"].cumsum()
)

frequency_scoring_analysis.style.format({
    "Customers": "{:,.0f}",
    "Customer Share": "{:.2%}",
    "Cumulative Share": "{:.2%}"
})


,Policies Purchased,Customers,Customer Share,Cumulative Share
0,1,"182,966",92.37%,92.37%
1,2,"8,319",4.20%,96.57%
2,3,"4,165",2.10%,98.68%
3,4,"1,996",1.01%,99.68%
4,5,559,0.28%,99.97%
5,6,62,0.03%,100.00%
6,7,6,0.00%,100.00%



#### Frequency Scoring

The Frequency distribution reveals a strong concentration of single-policy customers: 92.37% of Purchasing Customers have purchased exactly one policy during the observation period.

As a result, conventional quintile-based scoring would not provide five meaningful Frequency groups.

A discrete scoring approach is therefore adopted:

| Policies Purchased | F-Score |
|---|---|
| 1 | 1 |
| 2 | 2 |
| 3 | 3 |
| 4 | 4 |
| 5 or more | 5 |

This approach preserves the ordinal meaning of purchasing frequency without artificially splitting customers with identical purchase counts.

Importantly, a low Frequency score does not necessarily indicate poor customer quality or weak loyalty. In Wossist's insurance business, particularly for single-trip Travel products, a single observed purchase may represent a normal purchasing pattern.

Frequency is therefore interpreted as observed purchasing intensity rather than a standalone measure of customer loyalty.



#### Recency and Monetary Scoring

Unlike Frequency, Recency and Monetary Value exhibit greater variability across the customer base.

A percentile-based scoring approach is considered for both dimensions, dividing Purchasing Customers into five groups based on their observed values.

The scoring directions differ:

- **Recency:** lower values indicate more recent purchases and receive higher scores.
- **Monetary Value:** higher values indicate greater accumulated premium and receive higher scores.

Before implementing the scoring rules, the corresponding percentile thresholds are examined to verify that they provide meaningful and sufficiently distinct groups.


In [18]:

# Calculate percentile thresholds for Recency and Monetary Value
scoring_thresholds = pd.DataFrame({
    "Percentile": ["20%", "40%", "60%", "80%"],
    "Recency (days)": rfm_base["recency_days"]
        .quantile([0.20, 0.40, 0.60, 0.80]).values,
    "Monetary Value (€)": rfm_base["monetary_value"]
        .quantile([0.20, 0.40, 0.60, 0.80]).values
})

scoring_thresholds.style.format({
    "Recency (days)": "{:,.1f}",
    "Monetary Value (€)": "€{:,.2f}"
})


,Percentile,Recency (days),Monetary Value (€)
0,20%,72.0,€18.02
1,40%,168.0,€25.38
2,60%,267.0,€81.12
3,80%,527.0,€206.64


In [19]:

# Check distinct values and concentration for percentile-based scoring
scoring_feasibility = pd.DataFrame({
    "Dimension": ["Recency", "Monetary Value"],
    "Distinct Values": [
        rfm_base["recency_days"].nunique(),
        rfm_base["monetary_value"].nunique()
    ],
    "Most Common Value Share": [
        rfm_base["recency_days"].value_counts(normalize=True).iloc[0],
        rfm_base["monetary_value"].value_counts(normalize=True).iloc[0]
    ]
})

scoring_feasibility.style.format({
    "Most Common Value Share": "{:.2%}"
})


,Dimension,Distinct Values,Most Common Value Share
0,Recency,1075,0.81%
1,Monetary Value,49316,0.05%



### RFM Scoring Rules

The distribution analysis supports the adoption of a hybrid RFM scoring methodology.

Recency and Monetary Value are scored using customer-level quintile thresholds, while Frequency uses discrete thresholds due to the strong concentration of single-policy customers.

Each dimension receives a score between 1 and 5:

- **R-score:** 5 indicates the most recent purchasing behaviour.
- **F-score:** 5 indicates the highest observed purchasing frequency.
- **M-score:** 5 indicates the highest accumulated premium.

The observed percentile thresholds are:

| Percentile | Recency (days) | Monetary Value (€) |
|---|---:|---:|
| 20% | 72 | 18.02 |
| 40% | 168 | 25.38 |
| 60% | 267 | 81.12 |
| 80% | 527 | 206.64 |

Frequency scores follow the previously defined discrete classification.

The scoring system is intended to compare observed purchasing behaviour within the Wossist customer base. Scores are relative analytical indicators rather than direct measures of customer loyalty, profitability or future purchasing propensity.


In [20]:

# Calculate percentile thresholds for Recency and Monetary Value
r_thresholds = rfm_base["recency_days"].quantile(
    [0.20, 0.40, 0.60, 0.80]
).values

m_thresholds = rfm_base["monetary_value"].quantile(
    [0.20, 0.40, 0.60, 0.80]
).values

# Assign Recency scores (lower recency = higher score)
rfm_base["r_score"] = 5 - np.searchsorted(
    r_thresholds,
    rfm_base["recency_days"],
    side="left"
)

# Assign Frequency scores using discrete purchase counts
rfm_base["f_score"] = rfm_base["frequency"].clip(upper=5)

# Assign Monetary scores (higher monetary value = higher score)
rfm_base["m_score"] = 1 + np.searchsorted(
    m_thresholds,
    rfm_base["monetary_value"],
    side="left"
)

# Preview customer-level RFM scores
rfm_base.head()


,customer_id,recency_days,frequency,monetary_value,last_purchase_date,r_score,f_score,m_score
0,C000001,380,4,635.64,2025-12-16,2,4,5
1,C000002,887,1,162.95,2024-07-27,1,1,4
2,C000003,351,3,373.98,2026-01-14,2,3,5
3,C000004,973,1,18.46,2024-05-02,1,1,2
4,C000005,917,1,14.29,2024-06-27,1,1,1



### RFM Score Validation

After assigning the RFM scores, their distributions are examined to validate the scoring methodology and understand how customers are classified across the three dimensions.

The validation focuses on score completeness, the expected 1–5 range and the proportion of customers assigned to each score.

Recency and Monetary Value are expected to produce approximately balanced groups due to their percentile-based thresholds. Frequency, by contrast, is expected to remain strongly concentrated on the lowest score, reflecting the observed purchasing behaviour of Wossist customers.



In [21]:

# Calculate customer counts and shares for each RFM score
score_distributions = []

for dimension, column in {
    "Recency": "r_score",
    "Frequency": "f_score",
    "Monetary": "m_score"
}.items():

    counts = rfm_base[column].value_counts().sort_index()

    for score, count in counts.items():
        score_distributions.append({
            "Dimension": dimension,
            "Score": score,
            "Customers": count,
            "Customer Share": count / len(rfm_base)
        })

score_distribution = pd.DataFrame(score_distributions)

score_distribution.style.format({
    "Customers": "{:,.0f}",
    "Customer Share": "{:.2%}"
})


,Dimension,Score,Customers,Customer Share
0,Recency,1,"39,308",19.85%
1,Recency,2,"39,883",20.14%
2,Recency,3,"39,358",19.87%
3,Recency,4,"39,764",20.08%
4,Recency,5,"39,760",20.07%
5,Frequency,1,"182,966",92.37%
6,Frequency,2,"8,319",4.20%
7,Frequency,3,"4,165",2.10%
8,Frequency,4,"1,996",1.01%
9,Frequency,5,627,0.32%


In [22]:

# Validate score completeness and expected scoring range
score_columns = ["r_score", "f_score", "m_score"]

score_validation = pd.DataFrame({
    "Dimension": ["Recency", "Frequency", "Monetary"],
    "Missing Scores": [
        rfm_base[col].isna().sum() for col in score_columns
    ],
    "Invalid Scores": [
        (~rfm_base[col].isin([1, 2, 3, 4, 5])).sum()
        for col in score_columns
    ]
})

score_validation


,Dimension,Missing Scores,Invalid Scores
0,Recency,0,0
1,Frequency,0,0
2,Monetary,0,0



### RFM Scoring Validation Results

The validation confirms that all 198,073 Purchasing Customers have received valid RFM scores, with no missing or out-of-range values.

Recency and Monetary Value produce approximately balanced score groups, each representing around 20% of Purchasing Customers. Small differences are expected because customers with identical values are assigned the same score.

Frequency remains strongly concentrated on F-score 1 (92.37%), consistently with the observed predominance of single-policy customers.

These results confirm that the scoring rules have been applied as intended. However, the strong imbalance in Frequency must be considered when combining RFM dimensions into actionable customer segments.



## RFM Customer Segmentation

### Segmentation Approach

Traditional RFM segmentation evaluates customers through three dimensions:

- **Recency (R):** how recently a customer made a purchase.
- **Frequency (F):** how many purchases a customer made.
- **Monetary (M):** the total economic value generated by those purchases.

Common RFM segmentation frameworks combine these scores to identify customer groups such as Champions, Loyal Customers, Potential Loyalists, At Risk, and Hibernating. These classifications often emphasize Recency and Frequency to distinguish active, repeat, and potentially disengaged customers.

However, segmentation rules should reflect the purchasing patterns of the business being analyzed.

In Wossist, approximately **92% of purchasing customers have purchased only one policy**. This is consistent with a portfolio that includes occasional insurance needs, particularly single-trip Travel policies. Consequently, a conventional segmentation primarily based on Recency and Frequency would classify most customers according to purchase recency alone, while providing limited differentiation through Frequency.

Furthermore, insurance products can have substantially different premium amounts. Monetary therefore provides an important additional perspective, although it represents total premiums rather than profitability.

For these reasons, Wossist adopts a **business-adapted RFM segmentation**, using all three dimensions:

1. **Champions:** customers combining recent purchases, repeat purchasing, and high monetary value.
2. **High Value:** customers generating high premiums, regardless of purchase frequency.
3. **Repeat Customers:** customers with multiple purchases and lower monetary scores.
4. **Recency–Monetary profiles:** remaining customers classified by purchase recency and monetary value.

This approach preserves the RFM methodology while adapting segment definitions to the observed purchasing distribution.

### RFM Segment Definitions

Segments are assigned hierarchically, following the priority order below. Each customer belongs to exactly one segment.

| Segment | RFM Rule | Business Interpretation |
|---|---|---|
| **Champions** | R ≥ 4, F ≥ 2, M ≥ 4 | Recent, repeat purchasers generating high total premiums |
| **High Value** | M ≥ 4, excluding Champions | Customers generating relatively high total premiums |
| **Repeat Customers** | F ≥ 2, excluding previous segments | Repeat purchasers with lower monetary scores |
| **Recent Mid-Value** | R ≥ 4, M = 3, F = 1 | Recent single-purchase customers with intermediate premiums |
| **Recent Low-Value** | R ≥ 4, M ≤ 2, F = 1 | Recent single-purchase customers with lower premiums |
| **Less Recent Mid-Value** | R ≤ 3, M = 3, F = 1 | Less recent single-purchase customers with intermediate premiums |
| **Less Recent Low-Value** | R ≤ 3, M ≤ 2, F = 1 | Less recent single-purchase customers with lower premiums |

**Interpretation notes:**

- *High Value* and *Low-Value* refer to relative premium amounts, not customer profitability.
- *Less Recent* describes purchase timing and does not imply confirmed churn or inactivity.
- Purchase frequency is interpreted cautiously because a single policy can represent normal purchasing behavior in insurance.


In [30]:

# Assign business-adapted RFM segments using hierarchical rules

conditions = [
    (rfm_base["r_score"] >= 4)
    & (rfm_base["f_score"] >= 2)
    & (rfm_base["m_score"] >= 4),

    rfm_base["m_score"] >= 4,

    rfm_base["f_score"] >= 2,

    (rfm_base["r_score"] >= 4)
    & (rfm_base["m_score"] == 3),

    (rfm_base["r_score"] >= 4)
    & (rfm_base["m_score"] <= 2),

    (rfm_base["r_score"] <= 3)
    & (rfm_base["m_score"] == 3),

    (rfm_base["r_score"] <= 3)
    & (rfm_base["m_score"] <= 2)
]

segment_names = [
    "Champions",
    "High Value",
    "Repeat Customers",
    "Recent Mid-Value",
    "Recent Low-Value",
    "Less Recent Mid-Value",
    "Less Recent Low-Value"
]

rfm_base["rfm_segment"] = np.select(
    conditions,
    segment_names,
    default="Unclassified"
)

# Validate complete and exclusive assignment
assert (rfm_base["rfm_segment"] != "Unclassified").all()
assert rfm_base["customer_id"].is_unique

# Segment distribution
segment_distribution = (
    rfm_base["rfm_segment"]
    .value_counts()
    .reindex(segment_names, fill_value=0)
    .rename_axis("Segment")
    .reset_index(name="Customers")
)

segment_distribution["Customer Share"] = (
    segment_distribution["Customers"] / len(rfm_base)
)

display(
    segment_distribution.style.format({
        "Customers": "{:,.0f}",
        "Customer Share": "{:.1%}"
    })
)


,Segment,Customers,Customer Share
0,Champions,"6,575",3.3%
1,High Value,"72,654",36.7%
2,Repeat Customers,"3,598",1.8%
3,Recent Mid-Value,"13,063",6.6%
4,Recent Low-Value,"28,924",14.6%
5,Less Recent Mid-Value,"22,943",11.6%
6,Less Recent Low-Value,"50,316",25.4%



### RFM Segment Validation

The segment distribution provides an initial overview of customer allocation.

To assess whether the segments represent meaningful purchasing profiles, the following summary compares their average Recency, Frequency, Monetary value, and contribution to total written premiums.

Segment sizes are not expected to be uniform: the objective is to identify interpretable purchasing patterns rather than produce equally sized customer groups.


In [32]:

# Summarize purchasing behavior and premium contribution by RFM segment

segment_profile = (
    rfm_base
    .groupby("rfm_segment")
    .agg(
        customers=("customer_id", "nunique"),
        avg_recency_days=("recency_days", "mean"),
        avg_frequency=("frequency", "mean"),
        avg_monetary=("monetary_value", "mean"),
        total_premium=("monetary_value", "sum")
    )
    .reindex(segment_names)
    .reset_index()
)

segment_profile["customer_share"] = (
    segment_profile["customers"] / len(rfm_base)
)

segment_profile["premium_share"] = (
    segment_profile["total_premium"]
    / rfm_base["monetary_value"].sum()
)

display(
    segment_profile.style.format({
        "customers": "{:,.0f}",
        "avg_recency_days": "{:.0f}",
        "avg_frequency": "{:.2f}",
        "avg_monetary": "€{:,.2f}",
        "total_premium": "€{:,.0f}",
        "customer_share": "{:.1%}",
        "premium_share": "{:.1%}"
    })
)


,rfm_segment,customers,avg_recency_days,avg_frequency,avg_monetary,total_premium,customer_share,premium_share
0,Champions,"6,575",67,2.90,€383.54,"€2,521,772",3.3%,9.2%
1,High Value,"72,654",290,1.10,€298.44,"€21,683,190",36.7%,79.2%
2,Repeat Customers,"3,598",100,2.46,€51.99,"€187,072",1.8%,0.7%
3,Recent Mid-Value,"13,063",82,1.00,€42.46,"€554,680",6.6%,2.0%
4,Recent Low-Value,"28,924",83,1.00,€18.18,"€525,834",14.6%,1.9%
5,Less Recent Mid-Value,"22,943",428,1.00,€42.64,"€978,187",11.6%,3.6%
6,Less Recent Low-Value,"50,316",429,1.00,€18.20,"€915,994",25.4%,3.3%



### Key Findings & Business Interpretation

The resulting segmentation provides seven mutually exclusive customer groups, combining purchasing recency, repeat behavior, and total premium value.

The segmentation distinguishes:

- **High-value purchasing relationships**, including a more selective Champions group.
- **Repeat purchasing behavior**, independently of high monetary value.
- **Single-purchase customers**, differentiated through Recency and Monetary scores.

This distinction is particularly relevant for Wossist, where single-policy purchases represent the majority of purchasing relationships.

RFM segments describe historical purchasing behavior rather than predicting future purchases, customer profitability, or churn.

The resulting classification provides a foundation for subsequent Customer Business Profiling and Power BI analysis, where purchasing behavior can be interpreted alongside product, channel, lifecycle, and engagement characteristics.



## RFM Dataset Export

The final customer-level dataset contains the calculated RFM metrics, their respective scores, and the assigned RFM segment.

Only customers with at least one purchased policy are included in this dataset.

The exported file will be integrated into the subsequent Customer Business Profiling analysis, where RFM information will be joined to the complete customer population, including customers without purchases.


In [33]:

from pathlib import Path

# Select final customer-level RFM attributes
rfm_export = rfm_base[
    [
        "customer_id",
        "last_purchase_date",
        "recency_days",
        "frequency",
        "monetary_value",
        "r_score",
        "f_score",
        "m_score",
        "rfm_segment"
    ]
].copy()

# Export compressed CSV
output_dir = Path("output")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "customer_rfm.csv.gz"

rfm_export.to_csv(
    output_path,
    index=False,
    compression="gzip"
)

print(f"Export completed: {output_path}")
print(f"Customers exported: {len(rfm_export):,}")


Export completed: output\customer_rfm.csv.gz
Customers exported: 198,073



### Key Findings & Business Interpretation

The business-adapted RFM segmentation identifies seven distinct purchasing profiles across 198,073 customers.

**Key findings:**

- **Champions** represent 3.3% of purchasing customers but contribute 9.2% of total premiums. They combine recent purchases (67 days on average), repeat purchasing (2.90 policies per customer), and high monetary value (€383.54 on average).
- **High Value** customers account for 36.7% of purchasing customers and generate 79.2% of total premiums. Together with Champions, they represent approximately 40% of customers and 88.4% of premiums.
- **Repeat Customers** represent a smaller group (1.8%), characterized by multiple purchases (2.46 policies per customer on average) despite lower monetary values.
- **Recent and Less Recent Mid-/Low-Value segments** show consistent monetary profiles within their respective value tiers. Their main distinction is purchase recency: approximately 82–83 days for recent customers versus 428–429 days for less recent customers.

**Business interpretation:**

The results highlight the importance of combining Recency, Frequency, and Monetary rather than relying primarily on repeat purchasing behavior.

The segmentation separates economically significant customers, repeat purchasers, and single-purchase customers with different recency and monetary profiles.

These findings describe historical purchasing behavior. Premium amounts do not represent profitability, and less recent purchases do not necessarily indicate churn, particularly for occasional insurance products.

The resulting segments provide a foundation for subsequent Customer Business Profiling and Power BI analysis, where product, channel, lifecycle, and engagement characteristics can provide additional context.
